In [ ]:
!pip install langchain langgraph langchain-groq selenium beautifulsoup4 langchain-community langchain-huggingface chromadb sentence-transformers

In [ ]:
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = "api_key"

In [ ]:
from pathlib import Path
from langgraph.graph import StateGraph, END
from typing import TypedDict
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain.tools import tool
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from selenium import webdriver
from bs4 import BeautifulSoup

In [ ]:
def scrape_website(url):
    driver = webdriver.Chrome()
    driver.get(url)
    html = driver.page_source
    driver.quit()
    
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    return text

In [ ]:
# scrape_website("https://beautiful-soup-4.readthedocs.io/en/latest/#quick-start")

In [ ]:
llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    # other params...
)
def explain_content(text):
    prompt = f"""
    Explain the following website content in simple terms:

    {text}
    """
    return llm.invoke(prompt).content

In [ ]:
VECTOR_DB_DIR = (Path.cwd() / "vector_db")
VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma(
    collection_name="rag_docs",
    embedding_function=embedding_model,
    persist_directory=str(VECTOR_DB_DIR),
    collection_metadata={"hnsw:space": "cosine"},
)

print(f"Loaded RAG vector store from {VECTOR_DB_DIR}")

@tool
def scrape(url):
    """Scrape a webpage and return its content."""
    return scrape_website(url)

@tool
def explain(text):
    """Explain given text content."""
    return explain_content(text)

RAG_SCORE_THRESHOLD = 0.65

@tool
def rag(query: str, url_filter: str | None = None):
    """Answer a question using the local RAG vector store."""
    search_kwargs = {"k": 4}
    if url_filter:
        search_kwargs["filter"] = {"url": url_filter}

    matches = vectorstore.similarity_search_with_score(query, **search_kwargs)
    if not matches:
        return "I don't know based on the indexed data."

    relevant_matches = [
        (doc, score) for doc, score in matches if score <= RAG_SCORE_THRESHOLD
    ]
    if not relevant_matches:
        return "I don't know based on the indexed data."

    context_lines = []
    for rank, (doc, score) in enumerate(relevant_matches, start=1):
        source_url = doc.metadata.get("url", "unknown")
        context_lines.append(
            f"Source {rank} | url: {source_url} | score: {score:.4f}\n{doc.page_content}"
        )

    context_text = "\n\n".join(context_lines)
    prompt = f"""
    You are a strict RAG assistant.

    Answer ONLY using the context below.
    If the answer is not clearly present, say:
    "I don't know based on the indexed data."

    Question:
    {query}

    Context:
    {context_text}

    Answer:
    """
    return llm.invoke(prompt).content

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2740.73it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\light\AppData\Local\Temp\ipykernel_10588\1527944471.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Loaded RAG vector store from c:\Users\light\OneDrive\Desktop\intern\AI_agent\vector_db


In [ ]:
class AgentState(TypedDict):
    tools: str
    inputs: str
    last_task:str
    web_url: str
    scraped_content: str
    last_response: str
    task: str
    history: list
    memory_window: int 

In [ ]:
TOOLS = """
You have access to the following tools:

1. scrape(url: str)
   - Use this to extract content from a website URL.

2. explain(text: str)
   - Use this to explain or summarize given content.

3. rag(query: str)
   - Use this for factual questions about indexed pages or stored knowledge.
   - Prefer this over explain when the user is asking a question about a scraped page.
   - It should answer only from retrieved chunks in the vector database.


Decide which tool to use based on the user input.
If no tool is needed, return: finish
"""

In [ ]:

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

def scrape_node(state: AgentState):
    content = scrape.invoke({"url": state["web_url"]})

    metadata = {"url": state["web_url"], "source": "web_scrape"}
    docs = splitter.create_documents([content], metadatas=[metadata])

    vectorstore._collection.delete(where={"url": state["web_url"]})
    vectorstore.add_documents(docs)

    return {
        "scraped_content": content,
        "last_response": "Scraped page and indexed it in RAG with URL metadata."
    }

In [ ]:
def explain_node(state: AgentState):
    explanation = explain.invoke({"text": state["scraped_content"]})
    print(explanation)
    return {
        "last_response": explanation
    }

In [ ]:
def rag_node(state: AgentState):
    answer = rag.invoke({"query": state["inputs"], "url_filter": state.get("web_url")})
    print(answer)
    return {
        "last_response": answer
    }

In [ ]:
def rule_based_task(state: AgentState) -> str:
    user_text = state["inputs"].strip().lower()
    has_scraped_content = bool(state.get("scraped_content"))
    has_url_context = bool(state.get("web_url"))
    last_task = state.get("last_task", "")

    asks_history = any(phrase in user_text for phrase in ["previous", "earlier", "last response", "what did you say", "from history", "before"])
    asks_scrape = any(phrase in user_text for phrase in ["scrape", "fetch", "load", "index", "crawl"])
    asks_explain = any(phrase in user_text for phrase in ["explain", "summarize", "summary", "describe"])

    question_starts = ("what", "why", "how", "when", "where", "who", "which", "compare", "difference")
    is_question = "?" in user_text or user_text.startswith(question_starts)

    if asks_history:
        return "recall"
    if asks_explain and last_task == "scrape" and has_scraped_content:
        return "explain"
    if asks_scrape and last_task != "scrape":
        return "scrape"
    if is_question and (has_scraped_content or has_url_context):
        return "rag"
    if asks_explain and has_scraped_content:
        return "explain"
    return ""

def brain(state: AgentState):
    rule_choice = rule_based_task(state)
    if rule_choice:
        return {"task": rule_choice, "last_task": rule_choice}

    formatted_history = "\n".join([f"User: {t['user']} | Agent: {t['response']}" for t in state['history']])
    prompt = f"""
    You are an agent. Decisions: scrape, explain, rag, recall, finish.
    History: {formatted_history}
    Input: {state['inputs']}
    Last Task: {state['last_task']}
    Return one word only.
    """
    raw_decision = llm.invoke(prompt).content.strip().lower()
    decision = raw_decision.split()[0] if raw_decision else "finish"
    
    if decision not in {"scrape", "explain", "rag", "recall", "finish"}:
        decision = "finish"
        
    return {"task": decision, "last_task": decision}

In [ ]:
def recall_node(state: AgentState):
    formatted_history = []
    for turn in state['history']:
        formatted_history.append(
            f"User: {turn.get('user', '')} | Task: {turn.get('task', '')} | Agent: {turn.get('response', '')}"
        )
    formatted_history_text = "\n".join(formatted_history)

    prompt = f"""
You answer questions using the conversation memory below.

Conversation memory:
{formatted_history_text}

Current user question:
{state['inputs']}

Answer only from the stored memory. If the answer is not present, say you do not have that earlier response in memory.
"""

    answer = llm.invoke(prompt).content.strip()
    print(answer)
    return {
        "last_response": answer
    }

In [ ]:
def route_task(state: AgentState):
    print(state["task"], " called \n")
    return state["task"]

In [ ]:
builder = StateGraph(AgentState)

builder.add_node("brain", brain)
builder.add_node("scrape", scrape_node)
builder.add_node("explain", explain_node)
builder.add_node("rag", rag_node)
builder.add_node("recall", recall_node)

builder.set_entry_point("brain")

builder.add_conditional_edges(
    "brain",
    route_task,
    {
        "scrape": "scrape",
        "explain": "explain",
        "rag": "rag",
        "recall": "recall",
        "finish": END,
    },
)

builder.add_edge("scrape", "brain")   
builder.add_edge("explain", END)
builder.add_edge("rag", END)
builder.add_edge("recall", END)

graph = builder.compile()

In [ ]:
state = {
    "inputs": "",
    "tools": TOOLS,
    "web_url": "https://docs.langchain.com/oss/python/langchain/multi-agent/skills-sql-assistant",
    "scraped_content": "",
    "last_response": "",
    "task": "",
    "history": [],
    "last_task": "",
    "memory_window": 5
}

def run_agent(user_text, web_url = None):
    global state

    state["inputs"] = user_text
    if web_url is not None:
        state["web_url"] = web_url

    result = graph.invoke(state)
    state.update(result)

    executed_task = state.get("task", "finish")
    state["last_task"] = executed_task
    state["history"].append({
        "user": user_text,
        "task": executed_task,
        "response": state.get("last_response", ""),
    })

    window = state.get("memory_window", 5)
    state["history"] = state["history"][-window:]

    return state

In [30]:
run_agent("scrape this website and explain it too","https://en.wikipedia.org/wiki/Main_Page" )


scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 

scrape  called 



KeyboardInterrupt: 

In [ ]:
run_agent("explain this website")

explain  called 

Here’s a simple explanation of the website content you provided:

---

### **What is this website?**
This is the **main page of Wikipedia**, a free online encyclopedia where anyone can edit or add information. It’s run by volunteers and hosted by a non-profit group called the **Wikimedia Foundation**.

---

### **Key Features Explained**
1. **Navigation Tools**:
   - **Main Menu**: Links to sections like "Main Page," "Current Events," and "Random Article."
   - **Search Bar**: You can search for any topic.
   - **Appearance Settings**: Adjust font size, screen width, and light/dark mode.

2. **Featured Content**:
   - **Today’s Featured Article**: A comic book story called *"Sinestro Corps War"* (about superheroes vs. villains) is highlighted. It was popular and won awards.
   - **Recently Featured**: Fun facts about places (e.g., a sci-fi-themed restaurant), events (e.g., a snooker championship), and weather (e.g., a tropical storm).

3. **Random Fun Facts**:
   - Ex

{'inputs': 'explain this website',
 'tools': '\nYou have access to the following tools:\n\n1. scrape(url: str)\n   - Use this to extract content from a website URL.\n\n2. explain(text: str)\n   - Use this to explain or summarize given content.\n\n3. rag(query: str)\n   - Use this for factual questions about indexed pages or stored knowledge.\n   - Prefer this over explain when the user is asking a question about a scraped page.\n   - It should answer only from retrieved chunks in the vector database.\n\n\nDecide which tool to use based on the user input.\nIf no tool is needed, return: finish\n',
 'web_url': 'https://en.wikipedia.org/wiki/Main_Page',
 'scraped_content': 'Wikipedia, the free encyclopedia Jump to content Main menu Main menu move to sidebar hide Navigation Main page Contents Current events Random article About Wikipedia Contact us Contribute Help Learn to edit Community portal Recent changes Upload file Special pages Search Search Appearance Appearance move to sidebar hide

In [ ]:
run_agent("what is the difference between the main page and the talk page?")

rag  called 

The main page is the primary article or content page (e.g., Wikipedia's front page), while the talk page is a dedicated space for discussing improvements, issues, or questions related to the main page's content. As stated in the context, the talk page is explicitly for "discussing the contents of the Main Page," whereas the main page itself contains the article's actual content. General questions unrelated to the main page are directed elsewhere (e.g., the Teahouse).


{'inputs': 'what is the difference between the main page and the talk page?',
 'tools': '\nYou have access to the following tools:\n\n1. scrape(url: str)\n   - Use this to extract content from a website URL.\n\n2. explain(text: str)\n   - Use this to explain or summarize given content.\n\n3. rag(query: str)\n   - Use this for factual questions about indexed pages or stored knowledge.\n   - Prefer this over explain when the user is asking a question about a scraped page.\n   - It should answer only from retrieved chunks in the vector database.\n\n\nDecide which tool to use based on the user input.\nIf no tool is needed, return: finish\n',
 'web_url': 'https://en.wikipedia.org/wiki/Talk:Main_Page',
 'scraped_content': '',
 'last_response': 'The main page is the primary article or content page (e.g., Wikipedia\'s front page), while the talk page is a dedicated space for discussing improvements, issues, or questions related to the main page\'s content. As stated in the context, the talk pa